# 6. Evaluation Harness

Shared fraud-evaluation harness (FOC-176, phase F0).

This notebook is a smoke test for the helpers added to `funs.py`:
- `dataPreparation` with the FX-leakage fix (`amount_eur_fx_missing` flag)
- `chronological_split` (time-based, stratified fallback)
- `cross_validate_model` (StratifiedKFold, PR-AUC/ROC-AUC per fold)
- `evaluateModel` refactored (PR-AUC, ROC-AUC, F1, recall@precision, PR curve)

The feature set here is intentionally trivial (one-hot of `type`+`ccy`+
`customer_type`+`weekday`+`hour`) - it exists only to exercise the harness,
not as a tuned fraud model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from funs import (
    dataPreparation,
    chronological_split,
    cross_validate_model,
    evaluateModel,
)

## Data preparation

Load and prepare transactions. The FX-leakage fix means rows missing an
exchange rate now have `amount_eur = NaN` and `amount_eur_fx_missing = True`.

In [ ]:
data = dataPreparation(
    all_trxns_path="../data/all_trxns.csv",
    exchange_rates_path="../data/exchange_rates.csv",
)
print("rows:", len(data))
print("amount_eur_fx_missing:", int(data["amount_eur_fx_missing"].sum()))
print("fraud rate: %.4f" % (data["fraud_flag"].eq("Y").mean()))

## Build a trivial X/y

One-hot encode a few categorical columns as a smoke-test feature set.
`y` is the binary fraud flag (1 = fraud).

In [ ]:
feature_cols = ["type", "ccy", "customer_type", "weekday", "hour"]
X = pd.get_dummies(data[feature_cols], columns=feature_cols, drop_first=True)
y = data["fraud_flag"].eq("Y").astype(int)
print("X shape:", X.shape, " y positives:", int(y.sum()))

## Chronological split

Sort by `timestamp` and cut so the test set is strictly later than train.

In [ ]:
X_train, X_test, y_train, y_test = chronological_split(
    X, y, timestamp=data["timestamp"], test_size=0.2
)
print("train range:", data.loc[X_train.index, "timestamp"].min(), "->", data.loc[X_train.index, "timestamp"].max())
print("test  range:", data.loc[X_test.index, "timestamp"].min(), "->", data.loc[X_test.index, "timestamp"].max())
print("train positives:", int(y_train.sum()), " test positives:", int(y_test.sum()))

## Cross-validation

Stratified 5-fold CV on the full dataset, returning per-fold metrics + mean/std.

In [ ]:
cv_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
cv_result = cross_validate_model(cv_model, X, y, k=5, stratified=True)
print(cv_result["folds"])
print("Summary:")
for k, v in cv_result["summary"].items():
    print("  %s: %.4f" % (k, v))

## Evaluate (with probabilities)

Fit on the chronological train split, predict probabilities on the test split,
and call the refactored `evaluateModel` with `y_score`.

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

result = evaluateModel(y_test, y_pred, y_score=y_score, target_precision=0.5)
print("Result keys:", list(result.keys()))

In [ ]:
result